In [3]:
!pip install torch torchvision pillow matplotlib

In [ ]:
# Install required libraries
# pip install torch torchvision pillow matplotlib

import torch
import torch.optim as optim

from torchvision import models, transforms

from PIL import Image

import matplotlib.pyplot as plt


# Device configuration
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# Function to load image
def load_image(path, size=256):

    image = Image.open(path).convert("RGB")

    transform = transforms.Compose([

        transforms.Resize((size, size)),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],

            std=[0.229, 0.224, 0.225]
        )
    ])

    image = transform(image).unsqueeze(0)

    return image.to(device)


# Load content and style images
content = load_image("rabbit.jpg")

style = load_image("van_gogh.jpg")


# Load pretrained VGG19 model
vgg = models.vgg19(
    pretrained=True
).features.to(device).eval()


# Gram Matrix
def gram_matrix(tensor):

    b, c, h, w = tensor.size()

    tensor = tensor.view(c, h * w)

    return torch.mm(tensor, tensor.t())


# Extract features
def get_features(x):

    layers = {
        '0': 'conv1',
        '5': 'conv2',
        '10': 'conv3',
        '19': 'conv4',
        '28': 'conv5'
    }

    features = {}

    for name, layer in vgg._modules.items():

        x = layer(x)

        if name in layers:
            features[layers[name]] = x

    return features


# Content and style features
content_features = {

    k: v.detach()

    for k, v in get_features(content).items()
}

style_features = {

    k: v.detach()

    for k, v in get_features(style).items()
}


# Style gram matrices
style_grams = {

    layer: gram_matrix(style_features[layer])

    for layer in style_features
}


# Target image
target = content.clone().requires_grad_(True).to(device)


# Optimizer
optimizer = optim.Adam(
    [target],
    lr=0.003
)


# Loss weights
content_weight = 1e4

style_weight = 1e2


# Training loop
for i in range(100):

    target_features = get_features(target)

    # Content loss
    content_loss = torch.mean(

        (target_features['conv4']
         - content_features['conv4']) ** 2
    )

    # Style loss
    style_loss = 0

    for layer in style_grams:

        target_gram = gram_matrix(
            target_features[layer]
        )

        style_gram = style_grams[layer]

        style_loss += torch.mean(
            (target_gram - style_gram) ** 2
        )

    # Total loss
    total_loss = (
        content_weight * content_loss
        + style_weight * style_loss
    )

    optimizer.zero_grad()

    total_loss.backward()

    optimizer.step()

    # Clamp values
    with torch.no_grad():
        target.clamp_(0, 1)

    # Print progress
    if i % 50 == 0:
        print(
            f"Step {i}, "
            f"Loss: {total_loss.item()}"
        )


# Convert tensor to image
def im_convert(tensor):

    image = tensor.cpu().detach().clone()

    image = image.squeeze(0)

    image = image.permute(1, 2, 0)

    image = image * torch.tensor(
        [0.229, 0.224, 0.225]
    ) + torch.tensor(
        [0.485, 0.456, 0.406]
    )

    image = image.clamp(0, 1)

    return image


# Display final stylized image
plt.imshow(im_convert(target))

plt.title("Stylized Image")

plt.axis("off")

plt.show()

D:\ANACONDA\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
D:\ANACONDA\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to C:\Users\anike/.cache\torch\hub\checkpoints\vgg19-dcbb9e9d.pth


100%|███████████████████████████████████████████████████████████████████████████████| 548M/548M [00:40<00:00, 14.3MB/s]


Step 0, Loss: 313291374592.0
Step 50, Loss: 53017767936.0
